In [ ]:
!nvidia-smi

Thu May  7 10:19:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving random123.zip to random123.zip


In [ ]:
!unzip random123.zip -d /content/

Archive:  random123.zip
   creating: /content/random123/
  inflating: /content/random123/.gitignore  
  inflating: /content/random123/.travis.yml  
  inflating: /content/random123/GNUmakefile  
  inflating: /content/random123/LICENSE  
  inflating: /content/random123/README.md  
   creating: /content/random123/.git/
  inflating: /content/random123/.git/description  
   creating: /content/random123/.git/info/
  inflating: /content/random123/.git/info/exclude  
   creating: /content/random123/.git/branches/
   creating: /content/random123/.git/hooks/
  inflating: /content/random123/.git/hooks/post-update.sample  
  inflating: /content/random123/.git/hooks/pre-applypatch.sample  
  inflating: /content/random123/.git/hooks/push-to-checkout.sample  
  inflating: /content/random123/.git/hooks/pre-push.sample  
  inflating: /content/random123/.git/hooks/pre-merge-commit.sample  
  inflating: /content/random123/.git/hooks/pre-commit.sample  
  inflating: /content/random123/.git/hooks/pre-rebas

In [ ]:
!sed -i 's/philox4x32_10/philox4x32/g' cuda_philox.cu
!sed -i 's|GPU_BIN      := $(ROOT)/gpu_parallel/cuda_philox|GPU_BIN      := $(ROOT)/cuda_philox|' /content/Makefile
!sed -i 's|$(GPU_BIN): $(ROOT)/gpu_parallel/cuda_philox.cu|$(GPU_BIN): $(ROOT)/cuda_philox.cu|' /content/Makefile
!sed -i 's|R123_INC    := $(ROOT)/random123/include|R123_INC    := $(ROOT)/Random123/include|' /content/Makefile
!sed -i 's|R123_INC    := $(ROOT)/Random123/include|R123_INC    := $(ROOT)/random123/include|' /content/Makefile

In [ ]:
!nvcc -O2 -std=c++17 -arch=sm_75 \
-I/content/random123/include \
-o cuda_philox cuda_philox.cu

cuda_philox.cu(188): warning #177-D: function "read_baseline_time_seconds" was declared but never referenced
  static double read_baseline_time_seconds(const string& csv_path) {
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [ ]:
!make compile-gpu

  Compiling gpu_parallel/cuda_philox.cu...
nvcc -O2 -std=c++17 -I/content/random123/include -o /content/cuda_philox /content/cuda_philox.cu
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
/content/cuda_philox.cu(188): warning #177-D: function "read_baseline_time_seconds" was declared but never referenced
  static double read_baseline_time_seconds(const string& csv_path) {
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

  OK  cuda_philox


In [ ]:
!make run-gpu
!make run-roof
!make run-plots

mkdir -p /content/results
  -- GPU Philox CUDA
cd /content && /content/cuda_philox 300000000
  GPU Philox-4x32-10 CUDA PRNG
  CS-3006 Parallel and Distributed Computing

  GPU Device      : Tesla T4
  Compute Cap.    : 7.5
  SM count        : 40
  Total VRAM      : 14912 MB
  Free  VRAM      : 14807 MB
  Max threads/blk : 1024

  N               : 300,000,000
  Philox calls    : 75,000,000
  Threads / block : 256
  Grid blocks     : 292969
  Total threads   : 75,000,064

  Running 5 warm-up iterations...
  Running 5 timed iterations...

  Timed run 1/5  →  9.376 ms
  Timed run 2/5  →  9.360 ms
  Timed run 3/5  →  9.362 ms
  Timed run 4/5  →  9.363 ms
  Timed run 5/5  →  9.365 ms

  GPU baseline warmup (1 CUDA thread, 5 iters)...
  GPU baseline timed (5 iters)...
  Baseline run 1/5  →  10782.413 ms
  Baseline run 2/5  →  10783.229 ms
  Baseline run 3/5  →  10766.675 ms
  Baseline run 4/5  →  10759.930 ms
  Baseline run 5/5  →  10816.032 ms

  Running CPU single core Threefry benchmark (

In [ ]:
!sed -i 's|R123_INC    := $(ROOT)/Random123/include|R123_INC    := $(ROOT)/random123/include|' /content/Makefile
!sed -i 's|R123_INC    := $(ROOT)/random123/include|R123_INC    := $(ROOT)/random123/include|' /content/Makefile

!sed -i 's|GPU_BIN      := $(ROOT)/gpu_parallel/cuda_philox|GPU_BIN      := $(ROOT)/cuda_philox|' /content/Makefile
!sed -i 's|$(GPU_BIN): $(ROOT)/gpu_parallel/cuda_philox.cu|$(GPU_BIN): $(ROOT)/cuda_philox.cu|' /content/Makefile

!sed -i 's|compile: compile-baseline compile-mpi compile-bw compile-roof|compile: compile-gpu|' /content/Makefile
!sed -i 's|run: run-baseline run-mpi-1 run-mpi-2 run-mpi-4 run-mpi-8 run-mpi-10 run-bw run-roof run-plots|run: run-gpu|' /content/Makefile

!sed -i 's|$(NVCC) $(CXXFLAGS) -I$(R123_INC) -o $@ $<|$(NVCC) $(CXXFLAGS) -arch=sm_75 -I$(R123_INC) -o $@ $<|' /content/Makefile

In [ ]:
!ncu --version
!ncu --query-metrics | head

NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)
Device Tesla T4 (TU104)
--------------------------------------------------------------------------- --------------- --------------- ----------------------------------------------------------------------
Metric Name                                                                 Metric Type     Metric Unit     Metric Description                                                    
--------------------------------------------------------------------------- --------------- --------------- ----------------------------------------------------------------------
dram__bytes                                                                 Counter         byte            # of bytes accessed in DRAM                                           
dram__bytes_read                                                            Counter         byte            # of byt

In [ ]:
%%writefile cuda_philox_ncu.cu
#include <cuda_runtime.h>
#include <Random123/philox.h>
#include <iostream>
#include <cstdint>

#define CUDA_CHECK(call)                                      \
    do {                                                      \
        cudaError_t err = call;                               \
        if (err != cudaSuccess) {                             \
            std::cerr << "CUDA error: "                       \
                      << cudaGetErrorString(err)              \
                      << " at " << __FILE__ << ":"            \
                      << __LINE__ << std::endl;               \
            return 1;                                         \
        }                                                     \
    } while (0)

static constexpr int BLOCK_SIZE = 256;
static constexpr int PHILOX_OUTPUT_WORDS = 4;
static constexpr uint32_t KEY0 = 0xDEADBEEFu;
static constexpr uint32_t KEY1 = 0xCAFEBABEu;

__global__ void philox_generate_kernel(
    uint32_t* __restrict__ output,
    long long N,
    uint32_t key0,
    uint32_t key1)
{
    long long tid =
        static_cast<long long>(blockIdx.x) * blockDim.x + threadIdx.x;

    long long stride =
        static_cast<long long>(gridDim.x) * blockDim.x;

    long long total_calls =
        (N + PHILOX_OUTPUT_WORDS - 1) / PHILOX_OUTPUT_WORDS;

    philox4x32_key_t key = {{key0, key1}};

    for (long long call = tid; call < total_calls; call += stride) {
        philox4x32_ctr_t counter = {{
            static_cast<uint32_t>(call & 0xFFFFFFFFULL),
            static_cast<uint32_t>(call >> 32),
            0u,
            0u
        }};

        philox4x32_ctr_t result = philox4x32(counter, key);

        long long base = call * PHILOX_OUTPUT_WORDS;

        if (base + 0 < N) output[base + 0] = result.v[0];
        if (base + 1 < N) output[base + 1] = result.v[1];
        if (base + 2 < N) output[base + 2] = result.v[2];
        if (base + 3 < N) output[base + 3] = result.v[3];
    }
}

int main(int argc, char** argv)
{
    long long N = 10000000;
    if (argc >= 2) {
        N = std::stoll(argv[1]);
    }

    int device = 0;
    CUDA_CHECK(cudaSetDevice(device));

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));

    std::cout << "GPU: " << prop.name << "\n";
    std::cout << "N: " << N << "\n";

    size_t bytes = static_cast<size_t>(N) * sizeof(uint32_t);

    uint32_t* d_output = nullptr;
    CUDA_CHECK(cudaMalloc(&d_output, bytes));

    long long total_calls =
        (N + PHILOX_OUTPUT_WORDS - 1) / PHILOX_OUTPUT_WORDS;

    int threads_per_block = BLOCK_SIZE;
    int blocks = static_cast<int>(
        (total_calls + threads_per_block - 1) / threads_per_block
    );

    std::cout << "Blocks: " << blocks << "\n";
    std::cout << "Threads/block: " << threads_per_block << "\n";

    philox_generate_kernel<<<blocks, threads_per_block>>>(
        d_output, N, KEY0, KEY1
    );

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaFree(d_output));

    std::cout << "Done\n";
    return 0;
}

Writing cuda_philox_ncu.cu


In [ ]:
!nvcc -O2 -std=c++17 -arch=sm_75 \
-I/content/random123/include \
-o cuda_philox_ncu cuda_philox_ncu.cu

In [ ]:
!ncu --set roofline ./cuda_philox_ncu 300000000

==PROF== Connected to process 9342 (/content/cuda_philox_ncu)
GPU: Tesla T4
N: 300000000
Blocks: 292969
Threads/block: 256
==PROF== Profiling "philox_generate_kernel" - 0: 0%....50%....100% - 10 passes
Done
==PROF== Disconnected from process 9342
[9342] cuda_philox_ncu@127.0.0.1
  philox_generate_kernel(unsigned int *, long long, unsigned int, unsigned int) (292969, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.01
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    4,135,356
    Memory Throughput                 %        76.36
    DRAM Throughput                   %        76.36
    Duration                         ms         7.07
    L1/TEX Cache Throughput           %        64.60
 

In [ ]:
from google.colab import files
files.download('/content/results/gpu_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>